In [ ]:
# try:
#     from google.colab import drive
#     drive.mount('/content/drive')
# except:
#     pass

In [ ]:
import os

drive_folder_name = "ProductVoiceAnalytics"
os.chdir(f'/content/drive/MyDrive/Projects/GitHub/{drive_folder_name}')

!pwd

# Notebook 05 — Topic Intelligence


For a given ASIN:
1. Stream all its reviews from raw data
2. Embed with sentence-transformers
3. Cluster with BERTopic
4. Summarize each cluster into plain-English bullets via Claude API

Output: 5 praise themes + 5 complaint themes in plain English.

In [ ]:
# import os
# if os.path.basename(os.getcwd()) == 'notebooks':
#     os.chdir('..')

In [ ]:
# !pip install -r requirements.txt

In [ ]:
import gzip
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
import anthropic
from src.config import (
    RAW_REVIEWS_PATH,
    EMBED_MODEL,
    N_TOPICS,
    N_BULLETS,
)

## 1. Configuration

In [ ]:
# peek at a few ASINs from the raw file to find a valid one
import gzip, json

with gzip.open(RAW_REVIEWS_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        print(record.get('asin'), '|', record.get('reviewText', '')[:50])
        if i > 10:
            break

In [ ]:
TARGET_ASIN = '0060786817'

## 2. Stream Reviews for Target ASIN

In [ ]:
reviews = []

with gzip.open(RAW_REVIEWS_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        if record.get('asin') == TARGET_ASIN:
            text = record.get('reviewText', '').strip()
            if text:
                reviews.append(text)

print(f'Found {len(reviews)} reviews for ASIN {TARGET_ASIN}')

## 3. Embed Reviews

In [ ]:
embedder   = SentenceTransformer(EMBED_MODEL)
embeddings = embedder.encode(reviews, show_progress_bar=True)

print(f'Embeddings shape: {embeddings.shape}')

## 4. Cluster with BERTopic

In [ ]:
topic_model = BERTopic(nr_topics=N_TOPICS, language='english', calculate_probabilities=False)
topics, _   = topic_model.fit_transform(reviews, embeddings)

topic_info = topic_model.get_topic_info()
print(topic_info.head(N_TOPICS + 1))

## 5. Extract Representative Reviews per Topic

In [ ]:
def get_topic_reviews(reviews, topics, topic_id, n=10):
    indices = [i for i, t in enumerate(topics) if t == topic_id]
    sampled = [reviews[i] for i in indices[:n]]
    return sampled

valid_topic_ids = [t for t in set(topics) if t != -1]
topic_reviews   = {tid: get_topic_reviews(reviews, topics, tid) for tid in valid_topic_ids}

## 6. Summarize with Claude API

> This function gets extracted to `src/intelligence/summarizer.py` after the notebook works end-to-end.

In [ ]:
import os
from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic()

def summarize_topic(topic_reviews_list: list, sentiment: str) -> str:
    """
    Summarize a cluster of reviews into one plain-English bullet via Claude API.

    Args:
        topic_reviews_list: list of review strings for this topic
        sentiment: 'praise' or 'complaint'

    Returns:
        Single bullet string starting with an action verb
    """
    reviews_block = '\n'.join(topic_reviews_list)

    prompt = f"""You are analyzing Amazon Electronics product reviews.

The following reviews all share a common {sentiment} theme:

{reviews_block}

Summarize the core {sentiment} theme into ONE concise bullet point (max 20 words).
Start with an action verb.
Return only the bullet text, nothing else."""

    response = client.messages.create(
        model      = 'claude-haiku-4-5-20251001',
        max_tokens = 60,
        messages   = [{'role': 'user', 'content': prompt}]
    )

    return response.content[0].text.strip()

## 7. Generate Praise and Complaint Bullets

In [ ]:
# sort valid topics by cluster size descending
sorted_topics = sorted(
    valid_topic_ids,
    key=lambda tid: len(topic_reviews[tid]),
    reverse=True
)

praise_ids    = sorted_topics[:N_BULLETS]
complaint_ids = sorted_topics[N_BULLETS:N_BULLETS * 2]

praise_bullets    = [summarize_topic(topic_reviews[tid], 'praise')    for tid in praise_ids]
complaint_bullets = [summarize_topic(topic_reviews[tid], 'complaint') for tid in complaint_ids]

## 8. Display Results

In [ ]:
print(f'=== Product Intelligence: {TARGET_ASIN} ===')
print(f'Total reviews analysed: {len(reviews)}\n')

print('✅ Top Praise Themes')
for i, bullet in enumerate(praise_bullets, 1):
    print(f'  {i}. {bullet}')

print('\n⚠️  Top Complaint Themes')
for i, bullet in enumerate(complaint_bullets, 1):
    print(f'  {i}. {bullet}')